# 🏥 ArogyaAI Medical Triage Model Training

## Fine-tuning Bio_ClinicalBERT for Symptom Urgency Classification

This notebook trains an AI model to classify medical symptoms into three urgency levels:
- **Emergency** - Immediate medical attention required
- **Doctor Visit** - Consultation needed within 24-48 hours
- **Self-Care** - Can be managed at home

### Setup Instructions:
1. Open this notebook in Google Colab
2. Go to **Runtime → Change runtime type → T4 GPU → Save**
3. Run all cells sequentially (Ctrl+F9 or Runtime → Run all)
4. Training takes ~20-25 minutes

---

## Step 1: Install Required Packages

Installing all necessary libraries for model training.

In [1]:
# Install dependencies
!pip install transformers datasets torch scikit-learn pandas numpy
!pip install accelerate -U

print("✅ Installation complete!")

✅ Installation complete!


## Step 2: Import Libraries & Check GPU

Importing necessary libraries and verifying GPU availability.

In [2]:
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected - training will be slow!")

ModuleNotFoundError: No module named 'torch'

## Step 3: Mount Google Drive (Optional)

Mount Google Drive to save trained models permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create folder to save models
!mkdir -p /content/drive/MyDrive/ArogyaAI_Models

print("✅ Google Drive mounted!")
print("Models will be saved to: /content/drive/MyDrive/ArogyaAI_Models/")

## Step 4: Load Bio_ClinicalBERT Model

Loading the pre-trained Bio_ClinicalBERT model from HuggingFace. This model is already trained on clinical notes and understands medical terminology.

In [ ]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"

print("⏳ Loading Bio_ClinicalBERT from HuggingFace...")
print("This may take 1-2 minutes...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with classification head for 3 classes
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,  # 3 urgency levels: emergency (2), doctor (1), self-care (0)
    problem_type="single_label_classification"
)

# Move to GPU
model.to(device)

print("✅ Model loaded successfully!")
print(f"📊 Model size: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")
print(f"💾 Model memory: {sum(p.numel() * p.element_size() for p in model.parameters())/1e6:.1f} MB")

## Step 5: Create Training Dataset

Creating synthetic medical symptom data with urgency labels. In production, you would use real medical datasets.

In [ ]:
import random

# Base training data
training_data = [
    # EMERGENCY cases (label: 2)
    {"text": "severe chest pain radiating to left arm with sweating", "label": 2},
    {"text": "I can't breathe properly and feel like I'm choking", "label": 2},
    {"text": "sudden severe headache worst pain of my life", "label": 2},
    {"text": "unconscious and not responding to anything", "label": 2},
    {"text": "heavy bleeding that won't stop", "label": 2},
    {"text": "severe abdominal pain with vomiting blood", "label": 2},
    {"text": "chest tightness with difficulty breathing", "label": 2},
    {"text": "sudden weakness on one side of face and arm", "label": 2},
    {"text": "severe allergic reaction with throat swelling", "label": 2},
    {"text": "crushing chest pain with nausea", "label": 2},
    {"text": "cannot catch breath and chest hurts badly", "label": 2},
    {"text": "stroke symptoms with slurred speech", "label": 2},
    {"text": "heart attack symptoms radiating pain", "label": 2},
    {"text": "severe breathing difficulty and blue lips", "label": 2},
    {"text": "uncontrollable bleeding from injury", "label": 2},
    
    # DOCTOR visit cases (label: 1)
    {"text": "fever of 103F for 3 days not coming down", "label": 1},
    {"text": "persistent cough with yellow phlegm for week", "label": 1},
    {"text": "severe headache lasting several days", "label": 1},
    {"text": "stomach pain and diarrhea for 4 days", "label": 1},
    {"text": "infected wound with pus and redness", "label": 1},
    {"text": "high fever with body aches for 3 days", "label": 1},
    {"text": "persistent vomiting unable to keep food down", "label": 1},
    {"text": "ear pain with hearing loss", "label": 1},
    {"text": "skin rash spreading over body", "label": 1},
    {"text": "urinary tract infection symptoms", "label": 1},
    {"text": "fever with chills and weakness", "label": 1},
    {"text": "severe back pain limiting movement", "label": 1},
    {"text": "eye infection with discharge", "label": 1},
    {"text": "persistent fever over 101F", "label": 1},
    {"text": "swollen lymph nodes with sore throat", "label": 1},
    
    # SELF-CARE cases (label: 0)
    {"text": "mild headache after working long hours", "label": 0},
    {"text": "slight cold with runny nose", "label": 0},
    {"text": "minor muscle soreness from exercise", "label": 0},
    {"text": "mild fatigue and tiredness", "label": 0},
    {"text": "occasional sneezing with mild allergies", "label": 0},
    {"text": "slight cough started today", "label": 0},
    {"text": "minor stomach upset after meal", "label": 0},
    {"text": "mild sore throat in morning", "label": 0},
    {"text": "little bit of nausea", "label": 0},
    {"text": "mild fever 99F feeling okay otherwise", "label": 0},
    {"text": "minor headache with screen time", "label": 0},
    {"text": "slight tiredness need more sleep", "label": 0},
    {"text": "mild bloating after eating", "label": 0},
    {"text": "occasional acid reflux", "label": 0},
    {"text": "minor neck stiffness from bad posture", "label": 0},
]

# Data augmentation function - REDUCED to prevent overfitting
def augment_data(data, multiplier=8):
    """Create more training examples through augmentation (reduced multiplier to prevent overfitting)"""
    augmented = []
    
    for item in data:
        augmented.append(item)
        
        # Create variations (less aggressive than before)
        for _ in range(multiplier):
            text = item['text']
            
            # Add common prefixes (less frequently)
            if random.random() > 0.7:
                prefixes = ["I have ", "I'm experiencing ", "Suffering from ", "Having ", "My symptoms are "]
                text = random.choice(prefixes) + text
            
            # Add common suffixes (less frequently)
            if random.random() > 0.7:
                suffixes = [" please help", " need advice", " what should I do", " for 2 days", ""]
                text = text + random.choice(suffixes)
            
            # Synonym replacements (more diverse)
            if random.random() > 0.6:
                replacements = {
                    "severe": ["very bad", "intense", "extreme", "serious"],
                    "mild": ["slight", "minor", "light", "gentle"],
                    "persistent": ["continuous", "ongoing", "constant"],
                    "fever": ["high temperature", "elevated temperature"]
                }
                for old, new_list in replacements.items():
                    if old in text and random.random() > 0.6:
                        text = text.replace(old, random.choice(new_list))
            
            augmented.append({"text": text, "label": item['label']})
    
    return augmented

# Augment data with REDUCED multiplier (8 instead of 15)
print("⏳ Generating augmented training data...")
print("ℹ️  Using reduced augmentation to prevent overfitting")
augmented_data = augment_data(training_data, multiplier=8)

df = pd.DataFrame(augmented_data)

print(f"✅ Created {len(augmented_data)} training examples")
print(f"\n📊 Class distribution:")
print(df['label'].value_counts().sort_index())
print(f"\n   0 = Self-Care")
print(f"   1 = Doctor Visit")
print(f"   2 = Emergency")

# Show some examples
print(f"\n📝 Sample examples:")
for label in [0, 1, 2]:
    sample = df[df['label'] == label].sample(1).iloc[0]
    label_name = ['Self-Care', 'Doctor', 'Emergency'][label]
    print(f"\n{label_name}: \"{sample['text']}\"")

## Step 6: Prepare Dataset for Training

Splitting data into train/test sets and tokenizing for the model.

In [ ]:
# Split into train and test sets
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label']
)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128  # Maximum sequence length
    )

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenize datasets
print("\n⏳ Tokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("✅ Datasets prepared and tokenized!")
print(f"Train dataset: {len(train_dataset)} examples")
print(f"Test dataset: {len(test_dataset)} examples")

## Step 7: Configure Training Parameters

Setting up training hyperparameters and evaluation metrics.

In [ ]:
# Training configuration - OPTIMIZED to prevent overfitting
training_args = TrainingArguments(
    output_dir='/content/triage_model',           # Output directory
    num_train_epochs=3,                           # Reduced from 5 to 3 epochs to prevent overfitting
    per_device_train_batch_size=16,               # Batch size for training
    per_device_eval_batch_size=16,                # Batch size for evaluation
    learning_rate=2e-5,                           # Learning rate
    warmup_steps=50,                              # Reduced warmup steps
    weight_decay=0.05,                            # Increased from 0.01 to 0.05 for more regularization
    logging_dir='/content/logs',                  # Logging directory
    logging_steps=20,                             # Log every 20 steps
    eval_strategy="epoch",                        # Evaluate after each epoch
    save_strategy="epoch",                        # Save after each epoch
    load_best_model_at_end=True,                  # Load best model at end
    metric_for_best_model="accuracy",             # Metric to use
    greater_is_better=True,
    save_total_limit=2,                           # Keep only 2 best checkpoints
    report_to="none",                             # Disable wandb
    fp16=True,                                    # Mixed precision training (faster)
)

# Define evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    
    return {
        'accuracy': accuracy,
    }

print("✅ Training configuration complete!")
print("ℹ️  Anti-overfitting measures applied:")
print("   - Reduced epochs: 5 → 3")
print("   - Increased weight_decay: 0.01 → 0.05")
print("   - Reduced augmentation multiplier: 15 → 8")
print(f"\n📋 Training parameters:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Weight decay: {training_args.weight_decay}")
print(f"   Mixed precision (FP16): {training_args.fp16}")

## Step 8: Train the Model 🚀

Training the model - this will take approximately 10-15 minutes on T4 GPU.

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...")
print("=" * 60)
print("⏰ Estimated time: 10-15 minutes on T4 GPU")
print("=" * 60)

# Train the model
train_results = trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")
print("=" * 60)
print(f"\n📊 Training metrics:")
print(f"   Final loss: {train_results.training_loss:.4f}")
print(f"   Training time: {train_results.metrics['train_runtime']:.1f} seconds")

## Step 9: Evaluate Model Performance

Testing the trained model on unseen data and generating performance metrics.

In [ ]:
print("📊 Evaluating model on test set...\n")

# Get predictions
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Calculate accuracy
accuracy = accuracy_score(true_labels, pred_labels)
print("=" * 60)
print(f"✅ Test Accuracy: {accuracy*100:.2f}%")
print("=" * 60)

# Detailed classification report
label_names = ['Self-Care', 'Doctor', 'Emergency']
print("\n📋 Classification Report:")
print("-" * 60)
print(classification_report(true_labels, pred_labels, target_names=label_names, digits=3))

# Confusion matrix
print("\n🔢 Confusion Matrix:")
print("-" * 60)
cm = confusion_matrix(true_labels, pred_labels)
cm_df = pd.DataFrame(cm, columns=label_names, index=label_names)
print(cm_df)
print("\n(Rows = Actual, Columns = Predicted)")

# Per-class accuracy
print("\n✅ Per-class Accuracy:")
print("-" * 60)
for i, label_name in enumerate(label_names):
    class_correct = cm[i, i]
    class_total = cm[i, :].sum()
    class_accuracy = class_correct / class_total * 100
    print(f"{label_name:12s}: {class_accuracy:.1f}% ({class_correct}/{class_total})")

# Emergency recall (most critical metric)
emergency_recall = cm[2, 2] / cm[2, :].sum()
print(f"\n⚠️  Critical Metric - Emergency Recall: {emergency_recall*100:.1f}%")
print(f"   (Must be >95% for safety)")

## Step 10: Test with Real Examples

Testing the model with realistic symptom descriptions.

In [ ]:
def predict_urgency(text):
    """Predict urgency level for a symptom description"""
    # Tokenize input
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get prediction
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()
    
    labels = ['Self-Care', 'Doctor', 'Emergency']
    
    return {
        'urgency': labels[predicted_class],
        'confidence': confidence,
        'probabilities': {
            'Self-Care': predictions[0][0].item(),
            'Doctor': predictions[0][1].item(),
            'Emergency': predictions[0][2].item()
        }
    }

# Test with various symptom descriptions
print("🧪 Testing Model with Real Examples:")
print("=" * 80)

test_cases = [
    "I have severe chest pain and difficulty breathing",
    "Mild headache for a few hours after computer work",
    "High fever of 103F for 3 days not coming down with medicine",
    "Minor cold with slight runny nose",
    "Crushing pain in chest radiating to left arm and jaw",
    "Persistent cough with green phlegm for one week",
    "Slight stomach upset after eating spicy food",
    "Sudden weakness on right side of body and slurred speech",
    "Tired and fatigued, need more sleep",
    "Heavy bleeding that won't stop after injury"
]

for i, text in enumerate(test_cases, 1):
    result = predict_urgency(text)
    
    # Color coding for display
    urgency_emoji = {
        'Self-Care': '🟢',
        'Doctor': '🟡',
        'Emergency': '🔴'
    }
    
    print(f"\n{i}. Symptom: \"{text}\"")
    print(f"   {urgency_emoji[result['urgency']]} Urgency: {result['urgency']}")
    print(f"   Confidence: {result['confidence']*100:.1f}%")
    print(f"   Probabilities: Self-Care={result['probabilities']['Self-Care']*100:.0f}% | "
          f"Doctor={result['probabilities']['Doctor']*100:.0f}% | "
          f"Emergency={result['probabilities']['Emergency']*100:.0f}%")

print("\n" + "=" * 80)

## Step 11: Save Trained Model

Saving the fine-tuned model to Google Drive for later use in your backend.

In [ ]:
print("💾 Saving trained model...")

# Save to temporary location
model.save_pretrained('/content/triage_model_final')
tokenizer.save_pretrained('/content/triage_model_final')
print("✅ Saved to: /content/triage_model_final")

# Save to Google Drive (permanent storage)
model.save_pretrained('/content/drive/MyDrive/ArogyaAI_Models/triage_model')
tokenizer.save_pretrained('/content/drive/MyDrive/ArogyaAI_Models/triage_model')
print("✅ Saved to Google Drive: MyDrive/ArogyaAI_Models/triage_model")

# Model size
import os
def get_dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

model_size = get_dir_size('/content/triage_model_final') / (1024 * 1024)  # Convert to MB
print(f"\n📊 Model size: {model_size:.1f} MB")

# Create a zip file for download (optional)
print("\n📦 Creating downloadable zip file...")
!zip -r -q /content/triage_model.zip /content/triage_model_final
print("✅ Model zipped: /content/triage_model.zip")

# Uncomment to download directly
# from google.colab import files
# files.download('/content/triage_model.zip')

print("\n" + "=" * 60)
print("✅ Model training and saving complete!")
print("=" * 60)
print("\n📝 Next steps:")
print("   1. Download model from Google Drive")
print("   2. Integrate with your FastAPI backend")
print("   3. Test with your mobile app")
print("   4. Deploy to production!")

## Step 12: Load Saved Model (For Testing)

Loading the saved model to verify it works correctly.

In [ ]:
print("🔄 Loading saved model from Google Drive...")

# Load model and tokenizer
loaded_model = AutoModelForSequenceClassification.from_pretrained(
    '/content/drive/MyDrive/ArogyaAI_Models/triage_model'
)
loaded_tokenizer = AutoTokenizer.from_pretrained(
    '/content/drive/MyDrive/ArogyaAI_Models/triage_model'
)

loaded_model.to(device)
loaded_model.eval()  # Set to evaluation mode

print("✅ Model loaded successfully!")

# Quick test
test_text = "I have severe chest pain radiating to my arm"
inputs = loaded_tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]

label_names = ['Self-Care', 'Doctor', 'Emergency']
print(f"\n🧪 Quick Test:")
print(f"   Input: \"{test_text}\"")
print(f"   Prediction: {label_names[prediction]}")
print(f"   Confidence: {probabilities[prediction].item()*100:.1f}%")

print("\n✅ Model is ready for deployment!")

---

## 🎉 Training Complete!

### Summary of Results:

Your Bio_ClinicalBERT model has been fine-tuned for medical triage classification!

**Model Performance:**
- ✅ Accuracy: Expected 85-92%
- ✅ Emergency Recall: Should be >95% (critical for safety)
- ✅ Model Size: ~420 MB
- ✅ Training Time: ~10-15 minutes

**Model Location:**
- Google Drive: `MyDrive/ArogyaAI_Models/triage_model/`
- Temporary: `/content/triage_model_final/`

### Next Steps for Integration:

1. **Download the model** from Google Drive
2. **Copy to your backend** project folder
3. **Update FastAPI backend** to load and use this model
4. **Test integration** with your mobile app
5. **Add multilingual translation** layer
6. **Deploy to production**

### Backend Integration Code:

```python
# In your FastAPI backend (main.py)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load model at startup
model = AutoModelForSequenceClassification.from_pretrained('./triage_model')
tokenizer = AutoTokenizer.from_pretrained('./triage_model')

@app.post("/api/v1/symptom/analyze")
async def analyze_symptoms(report: SymptomReport):
    # Tokenize
    inputs = tokenizer(report.textInput, return_tensors="pt")
    
    # Predict
    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits).item()
    
    urgency_levels = ['self-care', 'doctor', 'emergency']
    return TriageResult(urgencyLevel=urgency_levels[prediction])
```

### Tips for Production:

- Add translation layer for multilingual support
- Implement rule-based safety override for critical symptoms
- Add confidence thresholds
- Monitor model performance in production
- Collect real user data for continuous improvement

---

**Questions or issues?** Check the HuggingFace documentation or reach out for help!

Happy Deploying! 🚀